In [13]:
#************************************************************
# Cell 1: Imports and Configuration
# Description: Import required libraries and set API keys
#************************************************************
import os
import requests

# Load API key from environment
price_api_key = os.getenv('POKE_PRICE_TRACKER_API_KEY')

print("Setup complete!")

Setup complete!


In [14]:
#************************************************************
# Cell 2: Search Function
# Description: Function to search for a Pokemon card by name and number
#************************************************************
def search_card(card_name, card_number):
    """
    Search for a Pokemon card and get its price
    
    Args:
        card_name: Name like "Rayquaza"
        card_number: Number like "#194" or "194"
    
    Returns:
        Card data with prices, or None if not found
    """
    # Remove # from number if it's there
    clean_number = card_number.replace('#', '')
    
    # API endpoint
    url = "https://www.pokemonpricetracker.com/api/prices"
    
    # Setup headers with API key
    headers = {
        'Authorization': f'Bearer {price_api_key}'
    }
    
    # Search parameters
    params = {
        'name': card_name,
        'number': clean_number,
        'includeHistory': 'true'
    }
    
    # Make the request
    response = requests.get(url, headers=headers, params=params)
    
    # Check if it worked
    if response.status_code != 200:
        print(f"Error: {response.status_code}")
        return None
    
    # Get the data
    data = response.json()
    results = data.get('data', [])
    
    if not results:
        print(f"No card found for {card_name} #{clean_number}")
        return None
    
    # Return first match
    return results[0]


In [15]:
#************************************************************
# Cell 3: Example Usage
# Description: Search for a card and display results
#************************************************************
# Search for a card
card_name = "Rayquaza"
card_number = "#194"

print(f"Searching for {card_name} {card_number}...")
print("-" * 50)

result = search_card(card_name, card_number)

if result:
    print(f"Found: {result['name']}")
    print(f"Number: {result['number']}")
    print(f"\nPrices:")
    
    # Show TCGPlayer prices
    if 'tcgplayer' in result:
        tcg_prices = result['tcgplayer'].get('prices', {})
        
        # Show holofoil prices (most common for rare cards)
        if 'holofoil' in tcg_prices:
            holo = tcg_prices['holofoil']
            print(f"  Market Price: ${holo.get('market', 'N/A')}")
            print(f"  Low: ${holo.get('low', 'N/A')}")
            print(f"  High: ${holo.get('high', 'N/A')}")
        
        # Show normal prices if no holofoil
        elif 'normal' in tcg_prices:
            normal = tcg_prices['normal']
            print(f"  Market Price: ${normal.get('market', 'N/A')}")
            print(f"  Low: ${normal.get('low', 'N/A')}")
            print(f"  High: ${normal.get('high', 'N/A')}")
    
    # Show eBay graded card prices (if available)
    if 'ebay' in result and 'prices' in result['ebay']:
        print(f"\neBay Graded Sales:")
        ebay_prices = result['ebay']['prices']
        
        # Debug: print all keys to see what's available
        print(f"  Available grades: {list(ebay_prices.keys())}")
        
        # Show PSA grades with average prices
        for grade, data in ebay_prices.items():
            if isinstance(data, dict) and 'stats' in data:
                avg_price = data['stats'].get('average')
                count = data['stats'].get('count', 0)
                if avg_price:
                    print(f"  PSA {grade}: ${avg_price:.2f} (based on {count} sales)")

Searching for Rayquaza #194...
--------------------------------------------------
Found: Rayquaza V
Number: 194

Prices:
  Market Price: $228.35
  Low: $184
  High: $299.97

eBay Graded Sales:
  Available grades: ['8', '9', '10']
  PSA 8: $183.85 (based on 3 sales)
  PSA 9: $202.62 (based on 3 sales)
  PSA 10: $556.31 (based on 3 sales)
